# Neural Identifier Training with Particle Filters - Lorenz System

In [33]:
import numpy as np
import plotly.graph_objects as go

In [34]:
# ============================================================
# 1) True nonlinear system (Lorenz System)
# ============================================================
def plant_dynamics(x, u, sigma=10.0, rho=28.0, beta=8.0/3.0):
    """
    Continuous dynamics for Lorenz system: x = [x, y, z]. 
    Returns x_dot.
    
    The Lorenz equations:
    dx/dt = σ(y - x)
    dy/dt = x(ρ - z) - y  
    dz/dt = xy - βz
    """
    x_state, y_state, z_state = x
    
    # Lorenz equations
    x_dot = sigma * (y_state - x_state)
    y_dot = x_state * (rho - z_state) - y_state
    z_dot = x_state * y_state - beta * z_state
    
    return np.array([x_dot, y_dot, z_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """
    One Euler step of the discrete plant with process noise.
    """
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

In [35]:
# ============================================================
# 2) RHONN structure
# ============================================================
def sigmoidal(z, beta=1.0):
    """Sigmoid S(z)."""
    z = np.clip(z, -500, 500)
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Features for a 3-state Lorenz system (no inputs):
    z = [S(x1), S(x2), S(x3), S(x1)S(x2), S(x1)S(x3), S(x2)S(x3), 
         S(x1)^2, S(x2)^2, S(x3)^2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # x
    s_x2 = sigmoidal(x_est[1])  # y
    s_x3 = sigmoidal(x_est[2])  # z
    
    return np.array([
        s_x1, s_x2, s_x3,                      # Linear terms
        s_x1*s_x2, s_x1*s_x3, s_x2*s_x3,      # Cross terms
        s_x1**2, s_x2**2, s_x3**2,            # Quadratic terms
        1.0                                     # Bias
    ])


def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predicts next-state component with a single RHONN neuron:
    x_i(k+1) = w_i^T z( x(k) , u(k) )
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

In [36]:
# ============================================================
# 3) EKF trainer over weights
# ============================================================
class EKF_RHONN_Trainer:
    """
    EKF on each neuron's weight vector, with random-walk weight dynamics.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One EKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        z_i = construct_z_vector(x_state_for_z)          # shape (num_features,)
        H_i = z_i.reshape(-1, 1)                          # column vector

        for i in range(self.num_neurons):
            # Predict covariance with regularization
            P_pred = self.P[i] + self.Q[i]
            
            # Add small regularization to maintain positive definiteness
            P_pred += np.eye(self.num_weights_per_neuron) * 1e-8

            # Innovation covariance (scalar) with improved numerical stability
            M_i = self.R[i][0] + (H_i.T @ P_pred @ H_i)[0, 0]
            if M_i < 1e-10:
                M_i = 1e-10

            # Predicted output for neuron i
            x_hat_pred_i = self.weights[i] @ z_i

            # Innovation: measured chi at k+1 minus prediction built from z at k
            e_i = chi_kp1[i] - x_hat_pred_i
            
            # Clip innovation to prevent extreme updates
            e_i = np.clip(e_i, -10.0, 10.0)

            # Kalman gain (flatten to 1D)
            K_i = (P_pred @ H_i).flatten() / M_i

            # Weight update with adaptive learning rate
            adaptive_eta = self.eta * (1.0 / (1.0 + np.abs(e_i) * 0.1))
            self.weights[i] += adaptive_eta * K_i * e_i

            # Joseph form covariance update for better numerical stability
            I_KH = np.eye(self.num_weights_per_neuron) - np.outer(K_i, H_i.ravel())
            self.P[i] = I_KH @ P_pred @ I_KH.T + np.outer(K_i, K_i) * self.R[i][0]
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.num_weights_per_neuron) * 1e-6

In [37]:
# ============================================================
# 4) Particle Filter trainer over weights
# ============================================================
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights)
    - Update (likelihood from chi_{k+1} vs prediction built with z at k)
    - ESS-triggered resampling
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=100,
                 initial_weights=None, Q_std=0.05, R_std=0.1, ess_threshold=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.1
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One PF step over all neuron weight-sets.

        chi_kp1: measured true states at k+1 (targets)
        chi_k  : measured true states at k   (for z)
        x_hat_previous: previous estimate at k (to complete z)
        """
        # Build z from time k (series-parallel)
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)
        z = construct_z_vector(x_state_for_z)  # (num_features,)

        # 1) Predict: random walk on weights
        for i in range(self.num_neurons):
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.Q_std

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Log-likelihood for stability
            ll = -0.5 * (innov**2) / self.R_var
            ll -= np.max(ll)
            like = np.exp(ll)

            self.weights_pf[i] *= like
            s = np.sum(self.weights_pf[i])
            if s < 1e-300:
                # Weight collapse safeguard
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= s

            # 3) Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)

    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        return [np.mean(self.particles[i], axis=0) for i in range(self.num_neurons)]


In [38]:
# ============================================================
# 4b) Unscented Kalman Filter (UKF) trainer over weights
# ============================================================
class UKF_RHONN_Trainer:
    """
    Unscented Kalman Filter on each neuron's weight vector.
    Uses sigma points to handle nonlinearities better than standard EKF.
    Critical: update uses chi_{k+1} as measurement; z is built from time k (series-parallel).
    """
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.0, 
                 alpha=1e-3, beta=2.0, kappa=None):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.eta = eta
        
        # UKF parameters
        self.alpha = alpha  # Spread of sigma points (typically 1e-4 to 1)
        self.beta = beta    # Prior knowledge about distribution (2 for Gaussian)
        self.kappa = kappa if kappa is not None else 3 - num_weights_per_neuron
        
        # Calculate lambda and weights for sigma points
        self.n = num_weights_per_neuron
        self.lambda_ = alpha**2 * (self.n + self.kappa) - self.n
        
        # Weights for mean and covariance computation
        self.Wm = np.zeros(2 * self.n + 1)
        self.Wc = np.zeros(2 * self.n + 1)
        
        self.Wm[0] = self.lambda_ / (self.n + self.lambda_)
        self.Wc[0] = self.lambda_ / (self.n + self.lambda_) + (1 - alpha**2 + beta)
        
        for i in range(1, 2 * self.n + 1):
            self.Wm[i] = 1.0 / (2 * (self.n + self.lambda_))
            self.Wc[i] = 1.0 / (2 * (self.n + self.lambda_))

        self.weights = []
        self.P = []
        self.Q = []
        self.R = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                w_i = np.copy(initial_weights[i])
            else:
                w_i = np.random.randn(num_weights_per_neuron) * 0.1
            self.weights.append(w_i)

            self.P.append(np.eye(num_weights_per_neuron) * P_init)
            self.Q.append(np.eye(num_weights_per_neuron) * Q_init)
            self.R.append(np.array([R_init]))

    def _generate_sigma_points(self, mean, covariance):
        """Generate sigma points for UKF."""
        n = len(mean)
        sigma_points = np.zeros((2 * n + 1, n))
        
        # First sigma point is the mean
        sigma_points[0] = mean
        
        # Calculate matrix square root
        try:
            sqrt = np.linalg.cholesky((n + self.lambda_) * covariance)
        except np.linalg.LinAlgError:
            # If Cholesky fails, use SVD
            U, s, Vh = np.linalg.svd(covariance)
            sqrt = U @ np.diag(np.sqrt(s)) @ Vh
            sqrt *= np.sqrt(n + self.lambda_)
        
        # Generate remaining sigma points
        for i in range(n):
            sigma_points[i + 1] = mean + sqrt[i]
            sigma_points[i + 1 + n] = mean - sqrt[i]
        
        return sigma_points

    def _measurement_function(self, weight_sigma_point, z_vector):
        """Measurement function: applies RHONN prediction with given weights."""
        return np.dot(weight_sigma_point, z_vector)

    def update(self, chi_kp1, chi_k, x_hat_previous):
        """
        One UKF update for all neurons.

        chi_kp1: np.array, measured true states at time k+1  (target)
        chi_k  : np.array, measured true states at time k    (for building z)
        x_hat_previous: np.array, previous estimate (time k) to complete z (series-parallel)
        """
        # Build series-parallel state for z: replace measured outputs at time k
        x_state_for_z = np.copy(x_hat_previous)
        x_state_for_z[0] = chi_k[0]  # x (measured output for Lorenz system)

        z_i = construct_z_vector(x_state_for_z)  # shape (num_features,)

        for i in range(self.num_neurons):
            # --- Prediction Step ---
            # Generate sigma points for current weights
            sigma_points = self._generate_sigma_points(self.weights[i], self.P[i])
            
            # Predict sigma points (weights don't change in prediction step for RHONN)
            predicted_sigma_points = sigma_points.copy()
            
            # Predict mean and covariance
            predicted_mean = np.sum(self.Wm[:, np.newaxis] * predicted_sigma_points, axis=0)
            
            predicted_cov = self.Q[i].copy()
            for j in range(2 * self.n + 1):
                diff = predicted_sigma_points[j] - predicted_mean
                predicted_cov += self.Wc[j] * np.outer(diff, diff)
            
            # --- Measurement Update ---
            # Transform sigma points through measurement function
            measurement_sigma_points = np.zeros(2 * self.n + 1)
            for j in range(2 * self.n + 1):
                measurement_sigma_points[j] = self._measurement_function(predicted_sigma_points[j], z_i)
            
            # Predicted measurement mean
            predicted_measurement = np.sum(self.Wm * measurement_sigma_points)
            
            # Innovation covariance
            innovation_cov = self.R[i][0]
            for j in range(2 * self.n + 1):
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                innovation_cov += self.Wc[j] * diff_meas**2
            
            # Cross-covariance
            cross_cov = np.zeros(self.n)
            for j in range(2 * self.n + 1):
                diff_state = predicted_sigma_points[j] - predicted_mean
                diff_meas = measurement_sigma_points[j] - predicted_measurement
                cross_cov += self.Wc[j] * diff_state * diff_meas
            
            # Kalman gain
            if innovation_cov < 1e-12:
                innovation_cov = 1e-12
            K = cross_cov / innovation_cov
            
            # Innovation
            innovation = chi_kp1[i] - predicted_measurement
            
            # State update
            self.weights[i] = predicted_mean + self.eta * K * innovation
            
            # Covariance update
            self.P[i] = predicted_cov - np.outer(K, K) * innovation_cov
            
            # Ensure positive definiteness
            self.P[i] = 0.5 * (self.P[i] + self.P[i].T)
            eigenvals = np.linalg.eigvals(self.P[i])
            if np.min(eigenvals) <= 0:
                self.P[i] += np.eye(self.n) * 1e-6

In [39]:
# ============================================================
# 5) Simulation
# ============================================================
if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # 'laplacian' | 'uniform' | 'gaussian'
    process_noise_std = 0.01

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [1.0, 1.0, 1.0]  # Initial conditions for Lorenz system
    u = 0.0

    # --- RHONN config ---
    num_neurons = 3  # Three states for Lorenz system
    num_features = 10  # Updated feature vector size for 3 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    # np.random.seed(12345)  # (optional) reproducibility of initial weights
    common_initial_weights = [np.random.uniform(-0.5, 0.5, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF --- (Tuned parameters for better Lorenz system performance)
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=5e-4, R_init=5e-3, P_init=0.5, eta=0.8
    )
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]

    # --- UKF ---
    ukf_trainer = UKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=1e-4, R_init=1e-2, P_init=1.0, eta=1.1,
        alpha=1e-3, beta=2.0  # UKF-specific parameters
    )
    x_hat_ukf = np.zeros((n_steps, 3))
    x_hat_ukf[0] = x_true[0]

    # --- PF ---
    n_particles = 800
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        Q_std=0.8, R_std=np.sqrt(0.001), ess_threshold=n_particles / 2  # ESS < N/2
    )

    # Force identical particle initialization if desired:
    def initialize_pf_with_common_weights(pf_trainer_instance, common_weights_list):
        for i in range(pf_trainer_instance.num_neurons):
            pf_trainer_instance.particles[i] = np.tile(
                common_weights_list[i], (pf_trainer_instance.n_particles, 1)
            )
            pf_trainer_instance.weights_pf[i] = np.ones(pf_trainer_instance.n_particles) / pf_trainer_instance.n_particles

    initialize_pf_with_common_weights(pf_trainer, common_initial_weights)

    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    print("Starting simulation...")
    for k in range(n_steps - 1):
        # ---- 1) true system -> k+1 ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ekf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ekf[k])

        x_state_for_z_ekf = np.copy(x_hat_ekf[k])
        x_state_for_z_ekf[0] = x_true[k][0]  # series-parallel uses measured x at k
        x_hat_ekf[k+1, 0] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[0])  # x
        x_hat_ekf[k+1, 1] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[1])  # y
        x_hat_ekf[k+1, 2] = RHONN_predict(x_state_for_z_ekf, ekf_trainer.weights[2])  # z

        # ---- 2b) UKF update (uses chi_{k+1} target, z from k), then predict x_hat_{k+1} ----
        ukf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_ukf[k])

        x_state_for_z_ukf = np.copy(x_hat_ukf[k])
        x_state_for_z_ukf[0] = x_true[k][0]  # series-parallel uses measured x at k
        x_hat_ukf[k+1, 0] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[0])  # x
        x_hat_ukf[k+1, 1] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[1])  # y
        x_hat_ukf[k+1, 2] = RHONN_predict(x_state_for_z_ukf, ukf_trainer.weights[2])  # z

        # ---- 3) PF update (chi_{k+1} vs z from k), then predict x_hat_{k+1} with mean weights ----
        pf_trainer.update(chi_kp1=x_true[k+1], chi_k=x_true[k], x_hat_previous=x_hat_pf[k])

        pf_weight_estimates = pf_trainer.get_estimate()
        x_state_for_z_pf = np.copy(x_hat_pf[k])
        x_state_for_z_pf[0] = x_true[k][0]  # series-parallel uses measured x at k
        x_hat_pf[k+1, 0] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[0])   # x
        x_hat_pf[k+1, 1] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[1])   # y
        x_hat_pf[k+1, 2] = RHONN_predict(x_state_for_z_pf, pf_weight_estimates[2])   # z

        if k % (n_steps // 10) == 0:
            print(f"Simulation progress: {k/n_steps*100:.1f}%")

    print("Simulation finished.")

Common Initial Weights:
  Neuron 0: [ 0.18249653 -0.0207923   0.4945881   0.42777573 -0.3273341  -0.2605369
 -0.32519256  0.14239062 -0.13069519  0.32856284]
  Neuron 1: [ 0.04114407 -0.32744976  0.15427577  0.48436724  0.44558362  0.14007665
  0.46882055  0.08569201 -0.28253874  0.34192536]
  Neuron 2: [-0.24761335  0.03282945  0.13100947  0.43975187  0.08363355  0.47363535
  0.43865905 -0.14949162  0.11599331 -0.0434464 ]
Starting simulation...
Simulation progress: 0.0%
Simulation progress: 10.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 20.0%
Simulation progress: 30.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 40.0%
Simulation progress: 50.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 60.0%
Simulation progress: 70.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation progress: 80.0%
Simulation progress: 90.0%
Simulation finished.
Simulation finished.


In [40]:
 # ============================================================
    # 6) Results & plots for Lorenz System
# ============================================================
mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)  # x
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # y
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)  # z

mse_x1_ukf = np.mean((x_true[:, 0] - x_hat_ukf[:, 0])**2)  # x
mse_x2_ukf = np.mean((x_true[:, 1] - x_hat_ukf[:, 1])**2)  # y
mse_x3_ukf = np.mean((x_true[:, 2] - x_hat_ukf[:, 2])**2)  # z

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)   # x
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)   # y
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # z

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    print(f"  Neuron {i+1} (x{i+1}): {ekf_trainer.weights[i]}")

print(f"\nFinal UKF-RHONN Weights:")
for i in range(3):
    print(f"  Neuron {i+1} (x{i+1}): {ukf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    print(f"  Neuron {i+1} (x{i+1}): {pf_estimates[i]}")

print("\n--- Performance Comparison (MSE) for Lorenz System ---")
print(f"EKF MSE x1 (x):         {mse_x1_ekf:.6f}")
print(f"EKF MSE x2 (y):         {mse_x2_ekf:.6f}")
print(f"EKF MSE x3 (z):         {mse_x3_ekf:.6f}")
print(f"UKF MSE x1 (x):         {mse_x1_ukf:.6f}")
print(f"UKF MSE x2 (y):         {mse_x2_ukf:.6f}")
print(f"UKF MSE x3 (z):         {mse_x3_ukf:.6f}")
print(f"PF  MSE x1 (x):         {mse_x1_pf:.6f}")
print(f"PF  MSE x2 (y):         {mse_x2_pf:.6f}")
print(f"PF  MSE x3 (z):         {mse_x3_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'Lorenz X State', 'y_label': 'X Value',
     'chi': 'χ₁ (True x)', 'x': 'x₁ (Est. x)'},
    {'idx': 1, 'var': 'y', 'desc': 'Lorenz Y State', 'y_label': 'Y Value',
     'chi': 'χ₂ (True y)', 'x': 'x₂ (Est. y)'},
    {'idx': 2, 'var': 'z', 'desc': 'Lorenz Z State', 'y_label': 'Z Value',
     'chi': 'χ₃ (True z)', 'x': 'x₃ (Est. z)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash', color='blue'))
    trace_ukf = go.Scatter(x=t_history, y=x_hat_ukf[:, i], mode='lines',
                        name=f"{state_info['x']} (UKF)", line=dict(dash='dashdot', color='green'))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot', color='red'))

    fig = go.Figure([trace_plant, trace_ekf, trace_ukf, trace_pf])
    fig.update_layout(
        title=f'Lorenz System RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states
error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x3_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_x1_ukf = x_true[:, 0] - x_hat_ukf[:, 0]
error_x2_ukf = x_true[:, 1] - x_hat_ukf[:, 1]
error_x3_ukf = x_true[:, 2] - x_hat_ukf[:, 2]

error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_x3_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ekf, mode='lines',
                        name=f'EKF Error x (MSE={mse_x1_ekf:.6f})', opacity=0.7, line=dict(color='blue')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ukf, mode='lines',
                        name=f'UKF Error x (MSE={mse_x1_ukf:.6f})', opacity=0.7, line=dict(color='green')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_pf, mode='lines',
                        name=f'PF Error x (MSE={mse_x1_pf:.6f})', opacity=0.7, line=dict(color='red')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ekf, mode='lines',
                        name=f'EKF Error y (MSE={mse_x2_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ukf, mode='lines',
                        name=f'UKF Error y (MSE={mse_x2_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_pf, mode='lines',
                        name=f'PF Error y (MSE={mse_x2_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dot')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_ekf, mode='lines',
                        name=f'EKF Error z (MSE={mse_x3_ekf:.6f})', opacity=0.7, line=dict(color='blue', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_ukf, mode='lines',
                        name=f'UKF Error z (MSE={mse_x3_ukf:.6f})', opacity=0.7, line=dict(color='green', dash='dash')))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_pf, mode='lines',
                        name=f'PF Error z (MSE={mse_x3_pf:.6f})', opacity=0.7, line=dict(color='red', dash='dash')))
fig2.update_layout(
    title='Lorenz System Identification Errors (EKF vs UKF vs PF)',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 3D Phase space plot
fig3d = go.Figure()
fig3d.add_trace(go.Scatter3d(x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2],
                           mode='lines', name='True Lorenz Attractor',
                           line=dict(color='black', width=3)))
fig3d.add_trace(go.Scatter3d(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2],
                           mode='lines', name='EKF Estimation',
                           line=dict(color='blue', width=2, dash='dash')))
fig3d.add_trace(go.Scatter3d(x=x_hat_ukf[:, 0], y=x_hat_ukf[:, 1], z=x_hat_ukf[:, 2],
                           mode='lines', name='UKF Estimation',
                           line=dict(color='green', width=2, dash='dashdot')))
fig3d.add_trace(go.Scatter3d(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2],
                           mode='lines', name='PF Estimation',
                           line=dict(color='red', width=2, dash='dot')))
fig3d.update_layout(
    title='Lorenz System - 3D Phase Space Comparison (EKF vs UKF vs PF)',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    font=dict(size=12)
)
fig3d.show()


Final EKF-RHONN Weights:
  Neuron 1 (x1): [ 4.91666886  3.85193932 -3.19642269 -2.36125832  8.92639743  8.9314918
 -4.92884449 -5.96966387  0.38639085 -8.27658741]
  Neuron 2 (x2): [ -4.17780958   1.52276883  -0.78276575  -1.18008307   6.11899984
  13.68771099  -3.05314642  -9.72907818   7.04927027 -10.07646239]
  Neuron 3 (x3): [-11.36637212  -8.74208276  11.3294394    4.05486359  -7.75885365
  -4.79102409   9.70899203   8.82070602  15.33808384   6.58188088]

Final UKF-RHONN Weights:
  Neuron 1 (x1): [ 8.1488476   8.2450032  -7.22340574 -1.80291681  3.7714091  -0.07087408
 -0.61464853 -0.67323693 -3.71832935 -1.53041834]
  Neuron 2 (x2): [-4.16134123  4.78502932 -0.29990371 -3.18886564  0.86019538  2.92979377
 -1.07390224  1.30226743 -0.15820641  0.49007723]
  Neuron 3 (x3): [ -5.10356141 -16.96192063  23.78867793   4.21233404  -7.54621262
   3.19266197   2.76432383   0.37917884   8.55961696   4.38205495]

Final PF-RHONN Weight Estimates:
  Neuron 1 (x1): [  3.8595092  -29.51714376  